# MATH840 Final project


# MATH840 Final Project: 48-Hour Electricity Generation Forecast

**Final model:** HarmARMA_K4 + Seasonal-Naive (50/50 blend) with an empirical 95% prediction interval

---

## 1. What is in the series?

The data are hourly electricity generation readings: **33,480 observations** from 2023-01-01 01:00 to 2026-10-27 00:00. The task is to forecast the next **48 hours** (2026-10-27 01:00 to 2026-10-29 00:00) with a 95% prediction interval.

**Data quality.** The series is complete. There are no duplicate timestamps, no missing values and no gaps, so no trimming or imputation was needed. Values range from 0.7 to 3,111 (mean 315.8, median 220.6, standard deviation 377.6). Some values are very close to zero, which makes MAPE unstable. This matters for the model comparison below.

**Patterns.**
- **Strong seasonality.** There is a pronounced daily cycle (24 h) and a weekly cycle (168 h), clearly visible in the ACF and seasonal profiles.
- **Structural change.** Before 2026 the mean level and the size of the daily swing were much smaller. From early 2026 the swing grows sharply, so the earlier years describe a different regime. I therefore **cropped the data at 2026-04-01** and trained and validated only on the more recent regime.
- **After the crop,** there is no distinct trend and the amplitude of the swing neither grows nor shrinks systematically.
- **Weather.** A weather forecast was provided but I did not use it (see Section 4).

---

## 2. How did you check the forecast before submitting it?

A single validation window can be misleading, so I used a **rolling-origin backtest**:

- **Origins:** midnight, every second day, from 2026-04-01 to 2026-10-24, giving **104 windows** of 48 hours.
- **Warm-up:** the first 30 windows are used only to build the empirical interval. The remaining **74 windows are scored.**
- **Benchmark:** every window is compared with the seasonal-naive forecast (repeat the last 24 hours).
- **Metrics:** RMSE, MAPE, Winkler score (interval quality) and empirical coverage. Models are compared as **ratios to seasonal-naive, window by window** (below 1 means better). This stops a few very large days from dominating the comparison.
- **No leakage:** each forecast uses only data before its origin, and each interval uses only windows that finished before that origin.

### Results (74 scored windows, ratio to seasonal-naive)

| Model | RMSE (geo. mean) | MAPE (geo. mean) | Winkler (geo. mean) | Coverage |
|---|---|---|---|---|
| **HarmARMA_K4 + SNaive** | **0.866** | **0.828** | **0.903** | 94.6% |
| HarmARMA_K6 + SNaive | 0.883 | 0.839 | 0.920 | 94.6% |
| HarmARMA_Weekly + SNaive | 0.905 | 0.847 | 0.966 | 94.8% |
| HarmARMA_K2 + SNaive | 0.914 | 0.862 | 0.929 | 94.1% |
| Blend (50% SNaive, 50% 7-day profile) | 0.928 | 0.974 | 0.982 | 95.1% |
| Seasonal-naive (benchmark) | 1.000 | 1.000 | 1.000 | 94.5% |

**Why this model won.**
- The chosen model has the lowest RMSE and Winkler score of all candidates. Its **RMSE ratio is 0.866, and its median window ratio is 0.847**.
- It **beats seasonal-naive in 77% of windows on RMSE** and **in 90.5% of windows on the combined rank score** (0.3·RMSE + 0.3·MAPE + 0.4·Winkler).
- In absolute terms, the average RMSE falls from **260.6 to 221.7 (−15%)**, the average MAPE from **209.5 to 161.0 (−23%)**, and the average Winkler score from **1,502 to 1,288 (−14%)**.
- It improves on **every aggregate measure** while keeping coverage at the nominal level (**94.6% against 95%**).
- On one metric, HarmARMA_K4 alone is better: its MAPE ratio is 0.767 against 0.828 for the blend. Blending with seasonal-naive costs some MAPE but gives clearly better RMSE, interval width and robustness, so I kept the blend.

**Model specification.** The model is a SARIMAX(2,0,1) with a constant and 4 daily Fourier harmonics as regressors. It is fitted on the **last 28 days** of **log(y + 20)** and back-transformed with a variance correction. I also tried the raw scale and Box-Cox. RMSE barely changed, but MAPE was slightly better on the log scale, which is consistent with the near-zero values noted above, so I kept the log scale. The 50/50 blend with seasonal-naive reduces the risk that the ARMA component drifts on unusual days.

---

## 3. How good is your interval?

The interval is **empirical and distribution-free**, so it makes no normality assumption:

1. For every earlier backtest window, I compute normalized errors **(actual − forecast) / (recent level + 20)**. The recent level is the mean of the previous 7 days.
2. I split the 48 forecast hours into **8 groups**: four parts of the day (night 21–05, ramp-up 06–10, peak 11–16, evening 17–20) × forecast day 1 or 2. This lets the interval be wider where and when the model is less certain.
3. In each group I take the **2.5% and 97.5% quantiles** of the normalized errors, rescale them by the current level and add them to the point forecast.
4. The lower bound is **clipped at 0**, since generation cannot be negative.

**Performance.**
- **Calibration:** empirical coverage is **94.6% against the 95% target**.
- **Compared with the benchmark's own interval:** the seasonal-naive interval covers only **82.3%** of actual values. It has a worse Winkler score (1,933) than the empirical interval gives HarmARMA_K4 + SNaive (1,288). The benchmark interval is both **less reliable and wider**, so the empirical approach is the better choice.
- **Final forecast:** the intervals use all 104 completed windows, and the current level (mean of the last 168 hours) is 362.9. For example, the first forecast hour (2026-10-27 01:00) is 78.1 with a 95% interval of [0, 253.5].

---

## 4. What is left, and what did not work?

**What did not work.**

| Approach | RMSE ratio vs SNaive | Comment |
|---|---|---|
| ETS (damped, 14-day / 28-day window) | 2.26 / 2.40 | Over 2× worse; even blended with SNaive it stays worse than the benchmark (1.47 / 1.51) |
| HarmARMA with differencing (d=1) | 2.28 | Differencing overreacts to the last observations and hurts forecasts |
| Profile means and medians (3 or 7 days) | 0.96–1.00 | Almost no gain in RMSE and a worse Winkler score (1.08–1.15) |
| Weekly harmonics (24 h + 168 h) | 1.06 (0.905 blended) | Extra parameters over a 28-day window add variance without improving accuracy |
| More harmonics (K=6) | 0.93 (0.883 blended) | Slightly worse than K=4, so K=4 is the sweet spot |

**What is left.**
- **Residual analysis.** I have not yet examined the residuals in detail. The ACF and PACF were used only to choose the ARMA order, not to test the final model's errors.
- **Weather data.** The provided weather forecast is a natural source of improvement, since weather likely drives much of the daily variation. I did not use it, so there is room for gains, especially on volatile days.
- **Adaptive blend weight.** The 50/50 blend is fixed. Weighting it by recent performance could improve it further.